* Use message prefilling and stop sequences only to get three different AWS commands in a single response
* There shouldn't be any comments or explanation
* Hint: message prefilling isn't limited to just characters like ```

In [17]:
import boto3
import json

In [18]:
messages = []

def add_messages(messages, role, content):
    messages.append({
        "role": role,
        "content": [
            {
                "text": content
            }
        ]
    })

def chat_with_claude(messages, stop_sequences=[]):
    client = boto3.client("bedrock-runtime", region_name="us-east-1")
    model_id = "global.anthropic.claude-sonnet-4-20250514-v1:0"
    
    response = client.converse_stream(
        modelId=model_id,
        messages=messages,
        inferenceConfig={
            "temperature": 0.2,
            "stopSequences": stop_sequences
        } 
    )
    
    text = ""
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta_text = event["contentBlockDelta"]["delta"]["text"]
            text += delta_text
    
    return text

In [20]:
messages = []

user_message = "Give three examples of AWS CLI commands for different AWS services as json objects with the following format: {\"service\": \"\", \"command\": \"\"}"
add_messages(messages, "user", user_message)

add_messages(messages, "assistant", "```json")

text = chat_with_claude(messages, stop_sequences=["```"])
#print("\n\n--- Raw text from stream ---")
#print(repr(text))
#print("--- End raw text ---\n")
text = json.loads(text.strip())
print(json.dumps(text, indent=2))



[
  {
    "service": "S3",
    "command": "aws s3 ls s3://my-bucket --recursive"
  },
  {
    "service": "EC2",
    "command": "aws ec2 describe-instances --instance-ids i-1234567890abcdef0"
  },
  {
    "service": "Lambda",
    "command": "aws lambda list-functions --region us-east-1"
  }
]
